# 🔀 Flat (Peer-to-Peer) Multi-Agent System

A lightweight LLM router reads the task and picks **one specialist agent** to handle it.  
All agents are peers — none has authority over the others.

```
 User Task
     │
     ▼
  [Router LLM]  ← structured output → one of: researcher | coder | writer
     │
     ▼
 [Chosen Agent] → ReAct loop (think → tool call → observe) → Answer
```

**Strengths:** Minimal overhead, very fast, easy to extend.  
**Limitations:** Only one agent runs per task — no collaboration or multi-step replanning.

---
**Install:**
```bash
uv sync --group notebooks
```
Set `OPENAI_API_KEY` in a `.env` file.

## ⚙️ Setup

In [1]:
from typing import Literal
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

load_dotenv()

# Single LLM instance shared by the router and all agents
llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("✅ Setup complete.")

✅ Setup complete.


## 🛠️ Tool Definitions

Each tool wraps a real-world capability. Stubs return mock strings here — swap in actual APIs or sandboxes for production.

In [2]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"[Web results for: {query}]"

@tool
def run_python(code: str) -> str:
    """Execute Python code and return the output."""
    return f"[Output of code: {code}]"

@tool
def write_file(filename: str, content: str) -> str:
    """Write content to a file and return confirmation."""
    return f"[Written to {filename}]"

print("✅ Tools defined.")

✅ Tools defined.


## 🤖 Agent Definitions

`create_agent` (from `langchain.agents`) builds a compiled agent runtime that can reason, call tools, and return a final response. It replaces the legacy `create_react_agent` import path.

The `system_prompt` parameter injects a **system message** that gives each agent its persona.

In [3]:
# Each agent gets its own system prompt and an exclusive tool set.
# create_agent returns a runnable agent that can call tools and emit final messages.

researcher = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt="You are a Researcher. Find and summarise information concisely.",
)

coder = create_agent(
    model=llm,
    tools=[run_python],
    system_prompt="You are a Coder. Write and execute Python code to solve problems.",
)

writer = create_agent(
    model=llm,
    tools=[write_file],
    system_prompt="You are a Writer. Produce clear, well-structured documents.",
)

AGENTS = {"researcher": researcher, "coder": coder, "writer": writer}
print("✅ Agents compiled:", list(AGENTS.keys()))

✅ Agents compiled: ['researcher', 'coder', 'writer']


## 🔀 Structured Router

`llm.with_structured_output` constrains the LLM to return a validated Pydantic object — no string parsing, no fallback guards needed. `Literal[...]` restricts the output to exactly our three agent names.

In [4]:
class RouteDecision(BaseModel):
    """The router's output: which agent should handle this task."""
    agent: Literal["researcher", "coder", "writer"]
    reason: str  # brief justification — useful for debugging / logging

# with_structured_output wraps the LLM so it always returns RouteDecision-shaped output
router_llm = llm.with_structured_output(RouteDecision)

ROUTER_SYSTEM = (
    "You are a task router. Given a task, choose the best agent:\n"
    "  - researcher: for finding or summarising information\n"
    "  - coder: for writing or running Python code\n"
    "  - writer: for producing documents, reports, or written content"
)


def run_flat_mas(task: str) -> str:
    """Route the task to a single agent and return its final answer."""
    # Step 1: structured routing with explicit validation for static type checkers
    decision_raw = router_llm.invoke([
        {"role": "system", "content": ROUTER_SYSTEM},
        {"role": "user",   "content": task},
    ])
    decision = RouteDecision.model_validate(decision_raw)
    print(f"\n[Router] → agent: '{decision.agent}'  reason: {decision.reason}")

    # Step 2: run the chosen agent
    agent   = AGENTS[decision.agent]
    result  = agent.invoke({"messages": [{"role": "user", "content": task}]})

    # The final answer is the content of the last message in the state
    return result["messages"][-1].content

## ▶️ Demo

The router should select **researcher** for this information-gathering task.

In [5]:
result = run_flat_mas("Find the latest Python 3.13 features and write a summary.")
print("\n=== FLAT MAS OUTPUT ===")
print(result)


[Router] → agent: 'researcher'  reason: The task involves finding the latest information about Python 3.13 features, which requires research skills to gather accurate and up-to-date data.

=== FLAT MAS OUTPUT ===
Python 3.13 introduces several new features and improvements:

1. **Performance Enhancements**: Python 3.13 includes optimizations that improve the performance of the language, making it faster and more efficient.

2. **Syntax and Language Features**: New syntax features have been added to make the language more expressive and easier to use.

3. **Standard Library Updates**: The standard library has been updated with new modules and improvements to existing ones, enhancing functionality and usability.

4. **Type Hints and Annotations**: Improvements in type hints and annotations provide better support for static type checking, aiding developers in writing more robust code.

5. **Security Improvements**: Enhanced security features have been implemented to protect against vulne

## 💡 Key Takeaways

- **`create_agent`** from `langchain.agents` is the current API for tool-using agents.
- **`llm.with_structured_output(Pydantic)`** replaces `prompt | llm | StrOutputParser()` + manual `.strip().lower()` for routing — outputs are always valid.
- Adding a new specialist: define its tools, call `create_agent`, add to `AGENTS`, and update `RouteDecision.agent`'s `Literal`.
- For tasks requiring **multiple sequential steps**, see `02_hierarchical_mas.ipynb`.